##Import Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

##Load Dataset

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

##Data Preprocessing

In [ ]:
# Convert images into tensors
X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
X_test = tf.convert_to_tensor(X_test, dtype=tf.float32)

# Flatten images (28x28 → 784)
X_train = tf.reshape(X_train, [-1, 28 * 28])
X_test = tf.reshape(X_test, [-1, 28 * 28])

# Normalize pixel values
X_train = X_train / 255.0
X_test = X_test / 255.0

# One-hot encoding labels
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

# Convert labels to tensors
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)

print("X_train tensor shape:", X_train.shape)
print("y_train tensor shape:", y_train.shape)

##Handling Missing Values

In [ ]:
print("Missing values in X_train:", tf.math.reduce_sum(
    tf.cast(tf.math.is_nan(X_train), tf.int32)
).numpy())

print("Missing values in X_test:", tf.math.reduce_sum(
    tf.cast(tf.math.is_nan(X_test), tf.int32)
).numpy())

print("Missing values in y_train:", tf.math.reduce_sum(
    tf.cast(tf.math.is_nan(y_train), tf.int32)
).numpy())

print("Missing values in y_test:", tf.math.reduce_sum(
    tf.cast(tf.math.is_nan(y_test), tf.int32)
).numpy())

#Build Neural Network Using TensorFlow Tensors
##Model Function

In [ ]:
def build_model(neurons=128, activation='relu'):

    model = tf.keras.Sequential([

        tf.keras.layers.Dense(
            neurons,
            activation=activation,
            input_shape=(784,)
        ),

        tf.keras.layers.Dense(
            64,
            activation=activation
        ),

        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Dense(
            10,
            activation='softmax'
        )
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

##Experiment 1 — ReLU

In [ ]:
print("Experiment 1: ReLU")

model1 = build_model(
    neurons=128,
    activation='relu'
)

history1 = model1.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

loss1, acc1 = model1.evaluate(X_test, y_test)

print("Experiment 1 Accuracy:", acc1)
print("Experiment 1 Loss:", loss1)

##Confusion Matrix — Experiment 1

In [ ]:
y_pred1 = model1.predict(X_test)

y_pred1_classes = np.argmax(y_pred1, axis=1)
y_true = np.argmax(y_test.numpy(), axis=1)

cm1 = confusion_matrix(y_true, y_pred1_classes)

disp = ConfusionMatrixDisplay(confusion_matrix=cm1)
disp.plot(cmap='Blues')

plt.title("Confusion Matrix - Exp 1")
plt.show()

##Experiment 2 — Tanh

In [ ]:
print("Experiment 2: Tanh")

model2 = build_model(
    neurons=128,
    activation='tanh'
)

history2 = model2.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

loss2, acc2 = model2.evaluate(X_test, y_test)

print("Experiment 2 Accuracy:", acc2)
print("Experiment 2 Loss:", loss2)

##Confusion Matrix — Experiment 2

In [ ]:
y_pred2 = model2.predict(X_test)

y_pred2_classes = np.argmax(y_pred2, axis=1)

cm2 = confusion_matrix(y_true, y_pred2_classes)

ConfusionMatrixDisplay(cm2).plot(cmap='Blues')

plt.title("Confusion Matrix - Exp 2")
plt.show()

##Experiment 3 — ReLU with 256 Neurons

In [ ]:
print("Experiment 3: ReLU 256")

model3 = build_model(
    neurons=256,
    activation='relu'
)

history3 = model3.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

loss3, acc3 = model3.evaluate(X_test, y_test)

print("Experiment 3 Accuracy:", acc3)
print("Experiment 3 Loss:", loss3)

##Confusion Matrix — Experiment 3

In [ ]:
y_pred3 = model3.predict(X_test)

y_pred3_classes = np.argmax(y_pred3, axis=1)

cm3 = confusion_matrix(y_true, y_pred3_classes)

ConfusionMatrixDisplay(cm3).plot(cmap='Blues')

plt.title("Confusion Matrix - Exp 3")
plt.show()

##Results Comparison

In [ ]:
print("FINAL COMPARISON")

print("{:<10} {:<15} {:<10} {:<10}".format(
    "Model",
    "Activation",
    "Accuracy",
    "Loss"
))

print("{:<10} {:<15} {:<10.4f} {:<10.4f}".format(
    "Model 1",
    "ReLU",
    acc1,
    loss1
))

print("{:<10} {:<15} {:<10.4f} {:<10.4f}".format(
    "Model 2",
    "Tanh",
    acc2,
    loss2
))

print("{:<10} {:<15} {:<10.4f} {:<10.4f}".format(
    "Model 3",
    "ReLU 256",
    acc3,
    loss3
))

##Visualization — Loss

In [ ]:
plt.plot(history1.history['loss'],
         label='Train Loss (ReLU 128)')

plt.plot(history1.history['val_loss'],
         label='Val Loss (ReLU 128)')


plt.plot(history2.history['loss'],
         label='Train Loss (Tanh 128)')

plt.plot(history2.history['val_loss'],
         label='Val Loss (Tanh 128)')


plt.plot(history3.history['loss'],
         label='Train Loss (ReLU 256)')

plt.plot(history3.history['val_loss'],
         label='Val Loss (ReLU 256)')


plt.title("Loss Comparison")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()

##Visualization — Accuracy

In [ ]:
plt.plot(history1.history['accuracy'],
         label='Train Acc (ReLU 128)')

plt.plot(history1.history['val_accuracy'],
         label='Val Acc (ReLU 128)')


plt.plot(history2.history['accuracy'],
         label='Train Acc (Tanh 128)')

plt.plot(history2.history['val_accuracy'],
         label='Val Acc (Tanh 128)')


plt.plot(history3.history['accuracy'],
         label='Train Acc (ReLU 256)')

plt.plot(history3.history['val_accuracy'],
         label='Val Acc (ReLU 256)')


plt.title("Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

# 🧠 Handwritten Digit Recognition using TensorFlow (MNIST)

---

## 📌 Project Overview
This project focuses on building a handwritten digit recognition system using a Multilayer Perceptron (MLP) implemented with TensorFlow/Keras.

The goal is to classify handwritten digits (0–9) from grayscale images using neural networks with high accuracy.

---

## 📊 Dataset Description
The MNIST dataset contains:

- 60,000 training images  
- 10,000 testing images  
- Each image is 28×28 pixels (grayscale)  
- Labels range from 0 to 9  

Each image represents a handwritten digit.

Dataset Link: https://keras.io/api/datasets/mnist/

---

## ⚙️ Data Preprocessing
The following preprocessing steps were applied:

✔ Flatten images from 28×28 → 784 vector  
✔ Normalize pixel values to range [0, 1]  
✔ One-hot encoding for labels using `to_categorical`  

### 🔄 TensorFlow and Tensors Usage
Although the dataset is initially loaded as NumPy arrays, the model is implemented using TensorFlow/Keras, which internally converts all computations into **TensorFlow tensors**.

This means that:
- All forward propagation operations  
- Loss calculations  
- Backpropagation and gradient updates  

are executed in tensor form for efficient computation and GPU acceleration.

The model was built using the Keras API, which is fully integrated with TensorFlow's tensor-based computation engine.

---

## 🧠 Model Architecture (MLP)
The neural network consists of:

- Input Layer: 784 neurons  
- Hidden Layer 1: 128 neurons (ReLU / Tanh)  
- Hidden Layer 2: 64 neurons  
- Dropout Layer: 0.3  
- Output Layer: 10 neurons (Softmax)

Implemented using TensorFlow Keras Sequential API.

---

## ⚙️ Training Configuration

- Optimizer: Adam  
- Loss Function: Categorical Crossentropy  
- Metrics: Accuracy  
- Batch Size: 32  
- Epochs: 10–20  
- Validation Split: 20%

Dataset Split:
- Training: 80%  
- Validation: 20% (from training data)  
- Testing: MNIST official test set  

---

## 🧪 Experiments

| Experiment | Activation | Neurons | Accuracy | Loss |
|------------|------------|----------|----------|------|
| Exp 1 | ReLU | 128 | 97.3% | 0.10 |
| Exp 2 | Tanh | 128 | 97.2% | 0.09 |
| Exp 3 | ReLU | 256 | 97.9% | 0.10 |

---

## 📈 Results and Analysis

✔ ReLU performed better than Tanh due to better gradient flow  
✔ Increasing neurons improved learning capacity  
✔ Dropout reduced overfitting and improved generalization  

---

## 🧠 Regularization Techniques

✔ Dropout (0.3) was used to reduce overfitting  

❌ Batch Normalization and Data Augmentation were not used because:

- MNIST is already clean and normalized  
- The model achieved high performance without them  

---

## 📊 Confusion Matrix

Used to evaluate classification performance:

- Diagonal values → correct predictions  
- Off-diagonal values → misclassifications  
- Helps identify commonly confused digits  

---

## 📊 Evaluation Metrics

- Accuracy → correct predictions  
- Loss → prediction error  
- Confusion Matrix → detailed class-level performance  

---

## ▶️ How to Run

```bash
pip install tensorflow matplotlib scikit-learn

## 📈 Analysis and Discussion

### Why ReLU outperforms Tanh:
ReLU does not saturate for positive values, which allows gradients to flow more efficiently during backpropagation.  
In contrast, Tanh saturates at both ends (output range between -1 and 1), which can lead to vanishing gradients in deeper networks and slower learning.

---

### Why 256 neurons (Exp 3) achieved higher accuracy:
Increasing the number of neurons from 128 to 256 increases the model capacity.  
This allows the network to learn more complex patterns from the 784 input pixels and extract richer features, improving generalization on the test set.

---

### Why Exp 3 trained for 20 epochs instead of 10:
Larger models require more epochs to fully converge.  
Training for 20 epochs allowed the model to optimize its weights properly and fully utilize its higher capacity without underfitting.

---

### Overall conclusion:
The best configuration was **Experiment 3 (ReLU, 256 neurons, 20 epochs)** with an accuracy of approximately **97.98%**.  
The combination of a non-saturating activation function and higher model capacity was the key factor in achieving the best performance.